# Notebook 04 — Territory Balancing

**Project:** APAC Territory Planning  
**Objective:** Measure how balanced APAC territories are today using the Gini coefficient, then simulate a rebalanced state and quantify the revenue impact.

**Business Question:** How should territories be realigned for next year, and what is the revenue impact of rebalancing?

In [1]:
import pandas as pd
import duckdb
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
import os

def gini_coefficient(values):
    """Calculate Gini coefficient for a list of values."""
    values = np.array(sorted(values))
    n = len(values)
    cumulative = np.cumsum(values)
    return (2 * np.sum((np.arange(1, n+1) * values))) / (n * cumulative[-1]) - (n + 1) / n

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

In [2]:
accounts      = pd.read_csv(f"{DATA_DIR}/accounts.csv")
reps          = pd.read_csv(f"{DATA_DIR}/reps.csv")
assignments   = pd.read_csv(f"{DATA_DIR}/assignments.csv")
opportunities = pd.read_csv(f"{DATA_DIR}/opportunities.csv")
customers     = pd.read_csv(f"{DATA_DIR}/customers.csv")
whitespace    = pd.read_csv(f"{PROCESSED_DIR}/whitespace_scored.csv")

con = duckdb.connect()
con.register("accounts",      accounts)
con.register("reps",          reps)
con.register("assignments",   assignments)
con.register("opportunities", opportunities)
con.register("customers",     customers)
con.register("whitespace",    whitespace)

print("Tables registered:")
print(con.execute("SHOW TABLES").df().to_string(index=False))

Tables registered:
         name
     accounts
  assignments
    customers
opportunities
         reps
   whitespace


## 1. Current Territory Balance — Gini Coefficient

In [3]:
sql = """
SELECT
    r.rep_id,
    r.rep_name,
    r.subregion,
    r.segment_focus,
    r.quota_usd,
    COUNT(DISTINCT a.account_id)        AS assigned_accounts,
    COALESCE(SUM(a.estimated_arr), 0)   AS territory_arr
FROM reps r
LEFT JOIN assignments asgn ON r.rep_id = asgn.rep_id
    AND asgn.coverage_status = 'Assigned'
LEFT JOIN accounts a ON asgn.account_id = a.account_id
WHERE r.subregion != 'Regional'
GROUP BY r.rep_id, r.rep_name, r.subregion, r.segment_focus, r.quota_usd
ORDER BY territory_arr DESC
"""

with open("../sql/territory_balance.sql", "w") as f:
    f.write(sql)

territory = con.execute(sql).df()

gini_arr     = gini_coefficient(territory["territory_arr"].values)
gini_accounts = gini_coefficient(territory["assigned_accounts"].values)

print("CURRENT TERRITORY DISTRIBUTION")
print(f"{'Rep':<22} {'Subregion':<16} {'Segment':<12} {'Accounts':>9} {'Territory ARR':>14}")
print("─" * 78)
for _, row in territory.iterrows():
    print(f"{row['rep_name']:<22} {row['subregion']:<16} {row['segment_focus']:<12} "
          f"{row['assigned_accounts']:>9,} {row['territory_arr']:>14,.0f}")

print()
print(f"Gini (Territory ARR):      {gini_arr:.3f}")
print(f"Gini (Assigned Accounts):  {gini_accounts:.3f}")
print()
print("Reference: 0.0 = perfectly equal | 0.2-0.3 = healthy | >0.4 = rebalancing needed")

CURRENT TERRITORY DISTRIBUTION
Rep                    Subregion        Segment       Accounts  Territory ARR
──────────────────────────────────────────────────────────────────────────────
Anthony Mccarty        Greater China    SMB                130     77,521,700
Michael Roberts        SEA              SMB                136     59,386,100
Arthur Miller          Greater China    SMB                132     57,087,000
Sara Ross              SEA              SMB                132     54,565,200
Jacob Wilson           ANZ              Mid-Market          74     50,635,100
Lauren Butler          SEA              SMB                132     48,892,600
Kayla Green            Greater China    SMB                111     48,531,100
Timothy Lawson         North Asia       Mid-Market          74     34,983,900
Mitchell Bowen         SEA              Mid-Market          75     25,927,100
Cameron Graham         India            SMB                 89     18,902,900
Lauren Underwood DDS   North Asi

## 2. Territory ARR vs Quota — Imbalance by Rep

In [4]:
territory = territory.merge(
    reps[["rep_id", "max_accounts"]], on="rep_id", how="left"
)
territory["load_pct"] = territory["assigned_accounts"] / territory["max_accounts"] * 100
territory["arr_share"] = territory["territory_arr"] / territory["territory_arr"].sum() * 100

print("TERRITORY BALANCE — ACCOUNTS VS CAPACITY")
print(f"{'Rep':<22} {'Subregion':<16} {'Segment':<12} {'Assigned':>9} {'Max':>5} {'Load%':>8} {'ARR Share':>10}")
print("─" * 88)
for _, row in territory.iterrows():
    overload = " ←" if row["load_pct"] > 200 else ""
    print(f"{row['rep_name']:<22} {row['subregion']:<16} {row['segment_focus']:<12} "
          f"{row['assigned_accounts']:>9,} {row['max_accounts']:>5} "
          f"{row['load_pct']:>7.1f}%{overload}  {row['arr_share']:>8.1f}%")

print()
gini_load = gini_coefficient(territory["load_pct"].values)
gini_arr  = gini_coefficient(territory["territory_arr"].values)
print(f"Gini (Load %):        {gini_load:.3f}")
print(f"Gini (Territory ARR): {gini_arr:.3f}")
print()
print("Reference: 0.0 = perfectly equal | 0.2-0.3 = healthy | >0.4 = rebalancing needed")

TERRITORY BALANCE — ACCOUNTS VS CAPACITY
Rep                    Subregion        Segment       Assigned   Max    Load%  ARR Share
────────────────────────────────────────────────────────────────────────────────────────
Anthony Mccarty        Greater China    SMB                130   150    86.7%      13.1%
Michael Roberts        SEA              SMB                136   150    90.7%      10.0%
Arthur Miller          Greater China    SMB                132   150    88.0%       9.6%
Sara Ross              SEA              SMB                132   150    88.0%       9.2%
Jacob Wilson           ANZ              Mid-Market          74    80    92.5%       8.5%
Lauren Butler          SEA              SMB                132   150    88.0%       8.2%
Kayla Green            Greater China    SMB                111   150    74.0%       8.2%
Timothy Lawson         North Asia       Mid-Market          74    80    92.5%       5.9%
Mitchell Bowen         SEA              Mid-Market          75    80 

## 3. Rebalancing Simulation

In [5]:
TARGET_LOAD = 0.85

rebalanced = territory.copy()
rebalanced["target_accounts"] = (rebalanced["max_accounts"] * TARGET_LOAD).astype(int)
rebalanced["excess_accounts"] = (
    rebalanced["assigned_accounts"] - rebalanced["target_accounts"]
).clip(lower=0)
rebalanced["rebalanced_accounts"] = (
    rebalanced["assigned_accounts"] - rebalanced["excess_accounts"]
)
rebalanced["rebalanced_load"] = (
    rebalanced["rebalanced_accounts"] / rebalanced["max_accounts"] * 100
)

avg_arr_per_account = territory["territory_arr"] / territory["assigned_accounts"]
rebalanced["rebalanced_arr"] = rebalanced["rebalanced_accounts"] * avg_arr_per_account

total_excess = rebalanced["excess_accounts"].sum()

# Headcount needed — how many new reps to absorb excess by segment
excess_by_segment = (
    rebalanced.groupby("segment_focus")
    .agg(excess=("excess_accounts", "sum"),
         max_accounts=("max_accounts", "mean"))
    .assign(reps_needed=lambda x: np.ceil(x["excess"] / (x["max_accounts"] * TARGET_LOAD)))
)

gini_before = gini_coefficient(rebalanced["load_pct"].values)
gini_after  = gini_coefficient(rebalanced["rebalanced_load"].values)

print(f"Total excess accounts after rebalancing to 85%: {total_excess:,.0f}")
print(f"These accounts need new headcount to be covered.")
print()
print("HEADCOUNT NEEDED BY SEGMENT")
print(f"{'Segment':<14} {'Excess Accounts':>16} {'Avg Capacity':>13} {'Reps Needed':>12}")
print("─" * 58)
for segment, row in excess_by_segment.iterrows():
    print(f"{segment:<14} {row['excess']:>16,.0f} {row['max_accounts']:>13,.0f} "
          f"{row['reps_needed']:>12,.0f}")

print()
print(f"Gini (Load %) — Before rebalancing:  {gini_before:.3f}")
print(f"Gini (Load %) — After rebalancing:   {gini_after:.3f}")
print()
print("REBALANCED TERRITORY — BEFORE VS AFTER")
print(f"{'Rep':<22} {'Subregion':<16} {'Before':>8} {'After':>8} {'Load Before':>12} {'Load After':>11}")
print("─" * 82)
for _, row in rebalanced.iterrows():
    print(f"{row['rep_name']:<22} {row['subregion']:<16} "
          f"{row['assigned_accounts']:>8,.0f} {row['rebalanced_accounts']:>8,.0f} "
          f"{row['load_pct']:>11.1f}% {row['rebalanced_load']:>10.1f}%")

Total excess accounts after rebalancing to 85%: 67
These accounts need new headcount to be covered.

HEADCOUNT NEEDED BY SEGMENT
Segment         Excess Accounts  Avg Capacity  Reps Needed
──────────────────────────────────────────────────────────
Enterprise                   11            40            1
Mid-Market                   29            80            1
SMB                          27           150            1

Gini (Load %) — Before rebalancing:  0.071
Gini (Load %) — After rebalancing:   0.052

REBALANCED TERRITORY — BEFORE VS AFTER
Rep                    Subregion          Before    After  Load Before  Load After
──────────────────────────────────────────────────────────────────────────────────
Anthony Mccarty        Greater China         130      127        86.7%       84.7%
Michael Roberts        SEA                   136      127        90.7%       84.7%
Arthur Miller          Greater China         132      127        88.0%       84.7%
Sara Ross              SEA        

## 4. Revenue Impact of Rebalancing

In [6]:
avg_customer_rate = len(customers) / len(assignments[assignments["coverage_status"] == "Assigned"])
avg_arr_per_customer = customers["arr"].mean()

ws_scored = pd.read_csv(f"{PROCESSED_DIR}/whitespace_scored.csv")
priority_accounts = ws_scored[ws_scored["priority_flag"] == "Priority"]
priority_arr_potential = priority_accounts["estimated_arr"].sum()

excess_revenue_potential = total_excess * avg_customer_rate * avg_arr_per_customer

print("REVENUE IMPACT ESTIMATE")
print("─" * 55)
print(f"Avg customer conversion rate:         {avg_customer_rate:>8.1%}")
print(f"Avg ARR per customer:                 {avg_arr_per_customer:>8,.0f}")
print()
print(f"Excess accounts needing coverage:     {total_excess:>8,.0f}")
print(f"Estimated ARR from excess accounts:   {excess_revenue_potential:>8,.0f}")
print()
print(f"Priority whitespace accounts:         {len(priority_accounts):>8,}")
print(f"Priority whitespace ARR potential:    {priority_arr_potential:>8,.0f}")
print()
print("Note: estimates based on observed conversion rate and avg ARR.")
print("Actual revenue depends on rep ramp time and market conditions.")

REVENUE IMPACT ESTIMATE
───────────────────────────────────────────────────────
Avg customer conversion rate:            26.5%
Avg ARR per customer:                   35,369

Excess accounts needing coverage:           67
Estimated ARR from excess accounts:    629,083

Priority whitespace accounts:              300
Priority whitespace ARR potential:    397,079,400

Note: estimates based on observed conversion rate and avg ARR.
Actual revenue depends on rep ramp time and market conditions.


## 5. Lorenz Curve — Territory Inequality Visualised

In [7]:
def lorenz_curve(values):
    values = np.sort(np.array(values, dtype=float))
    n = len(values)
    cumulative = np.cumsum(values)
    lorenz = np.concatenate([[0], cumulative / cumulative[-1]])
    uniform = np.linspace(0, 1, n + 1)
    return uniform, lorenz

x_load, y_load = lorenz_curve(rebalanced["load_pct"].values)
x_arr,  y_arr  = lorenz_curve(rebalanced["territory_arr"].values)

fig = go.Figure()

# Perfect equality line
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    name="Perfect Equality",
    line=dict(color="grey", dash="dash", width=1.5)
))

# Load % Lorenz curve
fig.add_trace(go.Scatter(
    x=x_load, y=y_load,
    mode="lines",
    name=f"Load % (Gini={0.071:.3f})",
    line=dict(color="#42A5F5", width=2.5)
))

# Territory ARR Lorenz curve
fig.add_trace(go.Scatter(
    x=x_arr, y=y_arr,
    mode="lines",
    name=f"Territory ARR (Gini={0.339:.3f})",
    line=dict(color="#1565C0", width=2.5)
))

fig.update_layout(
    title=dict(text="Lorenz Curve — Territory Load % vs ARR Distribution", y=0.95),
    xaxis=dict(title="Cumulative % of Reps", tickformat=".0%"),
    yaxis=dict(title="Cumulative % of Total", tickformat=".0%", scaleanchor="x", scaleratio=1),
    width=620,
    height=620,
    margin=dict(t=80, b=60, l=60, r=20),
    font=dict(size=13),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=0.95, x=0)
)

fig.show()
fig.write_image("../outputs/04_lorenz_curve.png", width=620, height=620, scale=2)
print("Saved: ../outputs/04_lorenz_curve.png")

Saved: ../outputs/04_lorenz_curve.png


## 6. Account Rebalancing List

In [8]:
# Logic: find accounts assigned to SMB reps that have high estimated_arr
# or Enterprise segment — these should be reassigned to Enterprise reps
# within the same subregion

# Get assignments with account and rep details
sql = """
SELECT
    a.account_id,
    a.company_name,
    a.subregion,
    a.segment,
    a.estimated_arr,
    a.employee_band,
    asgn.rep_id,
    r.rep_name        AS current_rep,
    r.segment_focus   AS current_rep_segment,
    r.max_accounts,
    asgn.engagement_status
FROM accounts a
JOIN assignments asgn ON a.account_id = asgn.account_id
JOIN reps r ON asgn.rep_id = r.rep_id
WHERE asgn.coverage_status = 'Assigned'
  AND a.is_customer = 0
"""

assigned_prospects = con.execute(sql).df()

# Flag mismatched accounts: Enterprise/high-ARR accounts with SMB reps
mismatch = assigned_prospects[
    (assigned_prospects["current_rep_segment"] == "SMB") &
    (
        (assigned_prospects["segment"] == "Enterprise") |
        (assigned_prospects["estimated_arr"] >= 250000)
    )
].copy()

# For each mismatched account find an Enterprise rep in same subregion
enterprise_reps = reps[
    (reps["segment_focus"] == "Enterprise") &
    (reps["subregion"] != "Regional")
][["rep_id", "rep_name", "subregion", "segment_focus"]].copy()

def find_recommended_rep(row):
    candidates = enterprise_reps[enterprise_reps["subregion"] == row["subregion"]]
    if candidates.empty:
        overlay = reps[reps["subregion"] == "Regional"]
        if not overlay.empty:
            rep = overlay.sample(1).iloc[0]
            return rep["rep_name"], "Regional Overlay"
        return "Unassigned", "No Coverage"
    rep = candidates.sample(1).iloc[0]
    return rep["rep_name"], rep["segment_focus"]

mismatch[["recommended_rep", "recommended_segment"]] = mismatch.apply(
    lambda x: pd.Series(find_recommended_rep(x)), axis=1
)

mismatch["reason"] = mismatch.apply(
    lambda x: f"Enterprise account (${x['estimated_arr']:,.0f} ARR) assigned to SMB rep", axis=1
)

rebalance_list = mismatch[[
    "account_id", "company_name", "subregion", "segment",
    "estimated_arr", "current_rep", "current_rep_segment",
    "recommended_rep", "recommended_segment", "reason"
]].sort_values("estimated_arr", ascending=False).reset_index(drop=True)

# Save to processed
rebalance_list.to_csv(f"{PROCESSED_DIR}/rebalancing_list.csv", index=False)

print(f"Accounts flagged for reassignment: {len(rebalance_list):,}")
print()
print("TOP 20 ACCOUNTS TO REASSIGN")
print(f"{'Company':<30} {'Subregion':<16} {'Segment':<12} {'Est ARR':>10} {'Current Rep':<22} {'Recommended Rep':<22}")
print("─" * 115)
for _, row in rebalance_list.head(20).iterrows():
    print(f"{row['company_name']:<30} {row['subregion']:<16} {row['segment']:<12} "
          f"{row['estimated_arr']:>10,.0f} {row['current_rep']:<22} {str(row['recommended_rep']):<22}")

print()
print(f"Full list saved to: data/processed/rebalancing_list.csv")
print()
print("Note: recommended rep assignments are illustrative — assigned randomly within subregion.")
print("Final assignments should consider rep capacity, industry expertise, and existing relationships.")

Accounts flagged for reassignment: 234

TOP 20 ACCOUNTS TO REASSIGN
Company                        Subregion        Segment         Est ARR Current Rep            Recommended Rep       
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Schwartz-Stewart               SEA              Enterprise    1,992,100 Michael Roberts        Haley Green           
Valenzuela and Sons            SEA              Enterprise    1,976,600 Michael Roberts        Michael Lynch         
King, Anderson and Mccarty     SEA              Enterprise    1,972,800 Michael Roberts        Michael Lynch         
Alexander, Walton and Peterson SEA              Enterprise    1,969,500 Sara Ross              Michael Lynch         
Bell-Hernandez                 SEA              Enterprise    1,967,000 Michael Roberts        Michael Lynch         
Chambers, Rivera and Chapman   SEA              Enterprise    1,962,400 Sara Ross              Haley Green  

## 7. Territory Balance Chart — Before vs After

In [9]:
fig = go.Figure()

fig.add_trace(go.Bar(
    name="Load % Before",
    x=rebalanced["rep_name"],
    y=rebalanced["load_pct"],
    marker_color="#90CAF9",
    text=[f"{v:.0f}%" for v in rebalanced["load_pct"]],
    textposition="outside"
))

fig.add_trace(go.Bar(
    name="Load % After",
    x=rebalanced["rep_name"],
    y=rebalanced["rebalanced_load"],
    marker_color="#1565C0",
    text=[f"{v:.0f}%" for v in rebalanced["rebalanced_load"]],
    textposition="outside"
))

fig.update_layout(
    title="Territory Load % — Before vs After Rebalancing",
    barmode="group",
    xaxis_title="Rep",
    xaxis_tickangle=-45,
    yaxis=dict(title="Load %", range=[0, 130]),
    height=550,
    font=dict(size=12),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)

fig.show()
fig.write_image("../outputs/04_territory_rebalancing.png", width=1100, height=550, scale=2)
print("Saved: ../outputs/04_territory_rebalancing.png")

Saved: ../outputs/04_territory_rebalancing.png


## Key Findings

1. **Territory ARR is unequal (Gini 0.339):** SMB reps hold $48-77M territory ARR while Enterprise reps hold only $11-13M — territory design favours volume over value
2. **Load is well distributed (Gini 0.071):** Reps have similar account counts but very different revenue potential — a territory design problem not a headcount problem
3. **Lorenz curve confirms the story:** Load % curve hugs the equality line, Territory ARR curve bows away — visual proof of structural imbalance
4. **Only 67 excess accounts after rebalancing to 85%:** 3 new reps (1 per segment) absorbs the overflow
5. **234 accounts flagged for reassignment:** Enterprise and high-ARR accounts currently assigned to SMB reps — full list saved to data/processed/rebalancing_list.csv
6. **India has no Enterprise rep:** 3 India Enterprise accounts routed to regional overlay reps — confirms overlay deployment recommendation
7. **Rebalancing alone unlocks $629K:** Priority whitespace of $397M requires new headcount and structured coverage model to unlock